# Mean Squared Error Gradient Descent

This notebook trains the course `mean_squared_error_gd` implementation on the processed data prepared in notebook 01.

The model is fitted on the full training split with its original class distribution. The prediction threshold is selected on the untouched validation split using F1, following notebook 01's baseline approach.

In [1]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'implementations.py').exists():
    PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from implementations import mean_squared_error_gd

In [2]:
DATA_PATH = PROJECT_ROOT / 'dataset/prepared/selected_random_over_dataset.npz'
data = np.load(DATA_PATH, allow_pickle=True)

x_train = data['x_train_raw']
y_train = data['y_train_raw'].astype(int)
x_test = data['x_test']
test_ids = data['test_ids']

tx = np.c_[np.ones((x_train.shape[0], 1)), x_train]
tx_test = np.c_[np.ones((x_test.shape[0], 1)), x_test]

print(f'Train shape: {tx.shape}')
print(f'Test shape: {tx_test.shape}')
print(f'Class counts: {np.bincount(y_train)}')

Train shape: (328135, 141)
Test shape: (109379, 141)
Class counts: [299160  28975]


In [3]:
def stratified_split(x, y, validation_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    fit_indices = []
    validation_indices = []
    for label in [0, 1]:
        indices = np.where(y == label)[0]
        rng.shuffle(indices)
        validation_size = int(len(indices) * validation_fraction)
        validation_indices.extend(indices[:validation_size])
        fit_indices.extend(indices[validation_size:])
    rng.shuffle(fit_indices)
    rng.shuffle(validation_indices)
    return x[fit_indices], y[fit_indices], x[validation_indices], y[validation_indices]


def metrics(y_true, y_pred):
    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    false_positive = np.sum((y_true == 0) & (y_pred == 1))
    false_negative = np.sum((y_true == 1) & (y_pred == 0))
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1


def best_threshold(y_true, scores):
    thresholds = np.unique(scores)
    f1_scores = [metrics(y_true, scores >= threshold)[2] for threshold in thresholds]
    return float(thresholds[np.argmax(f1_scores)])

In [ ]:
tx_fit, y_fit, tx_validation, y_validation = stratified_split(tx, y_train)

initial_w = np.zeros(tx_fit.shape[1])
max_iters = 10000
gamma = 0.2

weights, training_loss = mean_squared_error_gd(
    y_fit,
    tx_fit,
    initial_w,
    max_iters,
    gamma,
)
validation_scores = tx_validation @ weights
threshold = best_threshold(y_validation, validation_scores)
validation_predictions = (validation_scores >= threshold).astype(int)
precision, recall, f1 = metrics(y_validation, validation_predictions)

print(f'Training MSE: {training_loss:.4f}')
print(f'Validation threshold: {threshold:.4f}')
print(f'Validation precision: {precision:.4f}')
print(f'Validation recall: {recall:.4f}')
print(f'Validation F1: {f1:.4f}')

Training MSE: 0.0339
Validation threshold: 0.2035
Validation precision: 0.3278
Validation recall: 0.5379
Validation F1: 0.4074


In [5]:
# Retrain on the full training data after selecting the threshold.
final_weights, final_loss = mean_squared_error_gd(
    y_train,
    tx,
    np.zeros(tx.shape[1]),
    max_iters,
    gamma,
)

test_scores = tx_test @ final_weights
test_predictions = np.where(test_scores >= threshold, 1, -1)
print(f'Final training MSE: {final_loss:.4f}')
print(f'Predicted positive rate: {np.mean(test_predictions == 1):.4f}')

Final training MSE: 0.0342
Predicted positive rate: 0.1658


In [6]:
from helpers import create_csv_submission

submission_path = PROJECT_ROOT / 'submission_mse_gd.csv'
create_csv_submission(test_ids, test_predictions, submission_path)
print(f'Submission written to: {submission_path}')

Submission written to: /Users/mohamedzirar/Documents/year4/ML/ML_project1/submission_mse_gd.csv
